In [ ]:
# Bundle Analysis
# Analyzing cross-category bundle opportunities, optimal discounts, and CLV impact

#Economic Framework:
#- Complementarity analysis (not just association rules)
#- Bundle pricing optimization (profit maximization)
#- Customer heterogeneity (bundled vs unbundled buyers)
#- Revenue impact modeling (incremental lift)

# Loading and preparing the data
orders = pd.read_csv('../data/olist_orders_dataset.csv')
customers = pd.read_csv('../data/olist_customers_dataset.csv')

# Merging transactions with orders to get customer_id and order status
transactions = transactions.merge(
    orders[['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp']],
    on='order_id',
    how='left'
)

# Keeping only the delivered orders
transactions = transactions[transactions['order_status']=='delivered'].copy()

# Our 3 robust categories
focus_categories = ['watches_gifts', 'garden_tools', 'electronics']
category_names = {
    'watches_gifts': 'Watches/Gifts',
    'garden_tools': 'Garden Tools',
    'electronics': 'Electronics'
}

print(f"✅ Loaded {len(transactions):,} delivered transactions")
print(f"✅ {transactions['order_id'].nunique():,} unique orders")
print(f"✅ {transactions['customer_id'].nunique():,} unique customers")

In [ ]:
print("\n" + "="*80)
print("STEP 2: MARKET BASKET ANALYSIS")
print("="*80)

# Create order-category mapping
order_categories = transactions.groupby('order_id')['product_category_name_english'].apply(set).reset_index()
order_categories.columns = ['order_id', 'categories']

# Filter to orders containing at least one focus category
def contains_focus_category(cat_set):
    if pd.isna(cat_set):
        return False
    return any(cat in cat_set for cat in focus_categories)

order_categories['has_focus'] = order_categories['categories'].apply(contains_focus_category)
focus_orders = order_categories[order_categories['has_focus']].copy()

print(f"\nOrders containing our focus categories: {len(focus_orders):,}")

# Analyze basket composition
basket_stats = {
    'single_category': 0,
    'multi_category': 0,
    'focus_only': 0
}

for cats in focus_orders['categories']:
    focus_in_basket = [c for c in cats if c in focus_categories]
    if len(cats) == 1:
        basket_stats['single_category'] += 1
    else:
        basket_stats['multi_category'] += 1
    if len(focus_in_basket) == len(cats):
        basket_stats['focus_only'] += 1

print("\nBasket Composition:")
print(f"  Single category:  {basket_stats['single_category']:,} ({basket_stats['single_category']/len(focus_orders)*100:.1f}%)")
print(f"  Multi-category:   {basket_stats['multi_category']:,} ({basket_stats['multi_category']/len(focus_orders)*100:.1f}%)")
print(f"  Focus only:       {basket_stats['focus_only']:,} ({basket_stats['focus_only']/len(focus_orders)*100:.1f}%)")

In [ ]:
# ============================================================================
# STEP 3: COMPLEMENTARITY ANALYSIS (ECONOMIC FRAMEWORK)
# ============================================================================

print("\n" + "="*80)
print("STEP 3: COMPLEMENTARITY ANALYSIS")
print("="*80)

# Count co-occurrences
co_occurrence = {pair: 0 for pair in combinations(focus_categories, 2)}
single_occurrence = {cat: 0 for cat in focus_categories}

for cats in focus_orders['categories']:
    focus_in_basket = [c for c in cats if c in focus_categories]
    
    # Count singles
    for cat in focus_in_basket:
        single_occurrence[cat] += 1
    
    # Count pairs
    for pair in combinations(sorted(focus_in_basket), 2):
        if pair in co_occurrence:
            co_occurrence[pair] += 1

total_orders = len(focus_orders)

# Calculate support, confidence, and lift
bundle_metrics = []

for pair, count in co_occurrence.items():
    cat1, cat2 = pair
    
    # Support: P(A ∩ B)
    support = count / total_orders
    
    # Confidence: P(B|A) and P(A|B)
    conf_1_given_2 = count / single_occurrence[cat1] if single_occurrence[cat1] > 0 else 0
    conf_2_given_1 = count / single_occurrence[cat2] if single_occurrence[cat2] > 0 else 0
    
    # Lift: P(A ∩ B) / (P(A) × P(B))
    prob_1 = single_occurrence[cat1] / total_orders
    prob_2 = single_occurrence[cat2] / total_orders
    expected = prob_1 * prob_2
    lift = support / expected if expected > 0 else 0
    
    bundle_metrics.append({
        'category_1': category_names[cat1],
        'category_2': category_names[cat2],
        'co_purchases': count,
        'support': support,
        'confidence_1_2': conf_1_given_2,
        'confidence_2_1': conf_2_given_1,
        'lift': lift,
        'complementarity': 'Strong' if lift > 1.5 else 'Moderate' if lift > 1.2 else 'Weak'
    })

bundle_df = pd.DataFrame(bundle_metrics).sort_values('lift', ascending=False)

print("\nBundle Opportunity Analysis:")
print("-" * 80)
print(bundle_df.to_string(index=False))

# Save results
bundle_df.to_csv('../outputs/bundle_complementarity.csv', index=False)

In [ ]:
# ============================================================================
# STEP 4: CUSTOMER SEGMENTATION - BUNDLED VS UNBUNDLED
# ============================================================================

print("\n" + "="*80)
print("STEP 4: CUSTOMER SEGMENTATION ANALYSIS")
print("="*80)

# Identify bundle customers (bought 2+ focus categories in same order)
bundle_orders = []
for idx, row in focus_orders.iterrows():
    focus_in_basket = [c for c in row['categories'] if c in focus_categories]
    if len(focus_in_basket) >= 2:
        bundle_orders.append(row['order_id'])

# Get customer segments
bundle_order_ids = set(bundle_orders)
transactions['is_bundle_order'] = transactions['order_id'].isin(bundle_order_ids)

# Customer-level metrics
customer_metrics = transactions.groupby('customer_id').agg({
    'order_id': 'nunique',
    'price': 'sum',
    'is_bundle_order': 'max'  # 1 if ever bought bundle
}).reset_index()

customer_metrics.columns = ['customer_id', 'num_orders', 'total_spent', 'bought_bundle']

# Calculate CLV proxy (total spent / could use more sophisticated if had timestamps)
bundled_customers = customer_metrics[customer_metrics['bought_bundle'] == 1]
unbundled_customers = customer_metrics[customer_metrics['bought_bundle'] == 0]

print("\nCustomer Segmentation:")
print("-" * 80)
print(f"Bundle buyers:     {len(bundled_customers):,} ({len(bundled_customers)/len(customer_metrics)*100:.1f}%)")
print(f"Non-bundle buyers: {len(unbundled_customers):,} ({len(unbundled_customers)/len(customer_metrics)*100:.1f}%)")

print("\nAverage Metrics by Segment:")
print("-" * 80)
print(f"{'Metric':<25} {'Bundle Buyers':<20} {'Non-Bundle':<20} {'Lift':<15}")
print("-" * 80)

avg_bundle_orders = bundled_customers['num_orders'].mean()
avg_unbundle_orders = unbundled_customers['num_orders'].mean()
orders_lift = avg_bundle_orders / avg_unbundle_orders

avg_bundle_clv = bundled_customers['total_spent'].mean()
avg_unbundle_clv = unbundled_customers['total_spent'].mean()
clv_lift = avg_bundle_clv / avg_unbundle_clv

print(f"{'Orders per customer':<25} {avg_bundle_orders:<20.2f} {avg_unbundle_orders:<20.2f} {orders_lift:<15.2f}x")
print(f"{'CLV (total spent)':<25} BRL {avg_bundle_clv:<17.2f} BRL {avg_unbundle_clv:<17.2f} {clv_lift:<15.2f}x")

# Repeat purchase rate
bundled_repeat = (bundled_customers['num_orders'] > 1).sum() / len(bundled_customers) * 100
unbundled_repeat = (unbundled_customers['num_orders'] > 1).sum() / len(unbundled_customers) * 100

print(f"{'Repeat rate (%)':<25} {bundled_repeat:<20.1f} {unbundled_repeat:<20.1f} {bundled_repeat/unbundled_repeat:<15.2f}x")

In [ ]:
# ============================================================================
# STEP 5: BUNDLE DISCOUNT OPTIMIZATION
# ============================================================================

print("\n" + "="*80)
print("STEP 5: BUNDLE DISCOUNT OPTIMIZATION")
print("="*80)

# Get baseline pricing for top bundle opportunity
top_bundle = bundle_df.iloc[0]
cat1_key = [k for k, v in category_names.items() if v == top_bundle['category_1']][0]
cat2_key = [k for k, v in category_names.items() if v == top_bundle['category_2']][0]

# Get average prices
cat1_price = transactions[transactions['product_category_name_english'] == cat1_key]['price'].mean()
cat2_price = transactions[transactions['product_category_name_english'] == cat2_key]['price'].mean()
bundle_baseline_price = cat1_price + cat2_price

print(f"\nTop Bundle Opportunity: {top_bundle['category_1']} + {top_bundle['category_2']}")
print(f"  Lift: {top_bundle['lift']:.2f}x")
print(f"  Current co-purchase rate: {top_bundle['support']*100:.2f}%")
print(f"  Baseline price: BRL {bundle_baseline_price:.2f}")

# Estimate bundle demand elasticity (conservative assumption: -1.5)
bundle_elasticity = -1.5

# Test bundle discounts: 0%, 5%, 10%, 15%, 20%, 25%
discount_levels = np.array([0, 0.05, 0.10, 0.15, 0.20, 0.25])

# Baseline: current co-purchase count
baseline_bundle_orders = top_bundle['co_purchases']

# Assume cost structure (65% COGS as in main analysis)
cost_pct = 0.65
cat1_cost = cat1_price * cost_pct
cat2_cost = cat2_price * cost_pct
bundle_cost = cat1_cost + cat2_cost

# Calculate optimal discount
discount_results = []

for discount in discount_levels:
    bundle_price = bundle_baseline_price * (1 - discount)
    price_change = -discount
    
    # Demand response
    demand_multiplier = (1 + price_change) ** bundle_elasticity
    bundle_orders = baseline_bundle_orders * demand_multiplier
    
    # Revenue and profit
    bundle_revenue = bundle_price * bundle_orders
    bundle_profit = (bundle_price - bundle_cost) * bundle_orders
    
    # Current separate purchases profit (for comparison)
    baseline_profit = (bundle_baseline_price - bundle_cost) * baseline_bundle_orders
    
    discount_results.append({
        'discount_pct': discount * 100,
        'bundle_price': bundle_price,
        'bundle_orders': bundle_orders,
        'bundle_revenue': bundle_revenue,
        'bundle_profit': bundle_profit,
        'profit_vs_baseline': bundle_profit - baseline_profit,
        'profit_lift_pct': (bundle_profit / baseline_profit - 1) * 100
    })

discount_df = pd.DataFrame(discount_results)

# Find optimal discount
optimal_idx = discount_df['bundle_profit'].idxmax()
optimal = discount_df.iloc[optimal_idx]

print("\nBundle Discount Optimization Results:")
print("-" * 80)
print(discount_df.to_string(index=False, float_format=lambda x: f'{x:.2f}'))

print(f"\n🎯 OPTIMAL BUNDLE DISCOUNT: {optimal['discount_pct']:.0f}%")
print(f"   Bundle price: BRL {optimal['bundle_price']:.2f}")
print(f"   Expected orders: {optimal['bundle_orders']:.0f}")
print(f"   Expected profit: BRL {optimal['bundle_profit']:,.2f}")
print(f"   Profit lift: {optimal['profit_lift_pct']:+.1f}%")

# Save results
discount_df.to_csv('../outputs/bundle_discount_optimization.csv', index=False)
print("\n✅ Saved: bundle_discount_optimization.csv")

In [ ]:
# ============================================================================
# STEP 6: REVENUE IMPACT PROJECTION
# ============================================================================

print("\n" + "="*80)
print("STEP 6: TOTAL REVENUE IMPACT PROJECTION")
print("="*80)

# Project impact across all bundle opportunities
total_impact = 0
bundle_recommendations = []

for idx, bundle_row in bundle_df.iterrows():
    cat1_key = [k for k, v in category_names.items() if v == bundle_row['category_1']][0]
    cat2_key = [k for k, v in category_names.items() if v == bundle_row['category_2']][0]
    
    # Get prices
    c1_price = transactions[transactions['product_category_name_english'] == cat1_key]['price'].mean()
    c2_price = transactions[transactions['product_category_name_english'] == cat2_key]['price'].mean()
    bundle_price_full = c1_price + c2_price
    
    # Apply optimal discount (use 10% as standard)
    discount = 0.10
    bundle_price = bundle_price_full * (1 - discount)
    
    # Cost
    bundle_cost_total = (c1_price + c2_price) * cost_pct
    
    # Current co-purchases
    current_bundles = bundle_row['co_purchases']
    
    # Projected increase (conservative: 30% adoption lift from discount)
    adoption_lift = 1.30
    projected_bundles = current_bundles * adoption_lift
    
    # Profit calculation
    current_profit = (bundle_price_full - bundle_cost_total) * current_bundles
    projected_profit = (bundle_price - bundle_cost_total) * projected_bundles
    incremental_profit = projected_profit - current_profit
    
    total_impact += incremental_profit
    
    bundle_recommendations.append({
        'bundle': f"{bundle_row['category_1']} + {bundle_row['category_2']}",
        'lift': bundle_row['lift'],
        'current_bundles': current_bundles,
        'projected_bundles': projected_bundles,
        'recommended_discount': '10%',
        'bundle_price': bundle_price,
        'incremental_profit': incremental_profit
    })

bundle_rec_df = pd.DataFrame(bundle_recommendations)

print("\nBundle Recommendations:")
print("-" * 80)
print(bundle_rec_df.to_string(index=False))

print(f"\n💰 TOTAL INCREMENTAL PROFIT FROM BUNDLING: BRL {total_impact:,.2f}")

# Save recommendations
bundle_rec_df.to_csv('../outputs/bundle_recommendations.csv', index=False)
print("✅ Saved: bundle_recommendations.csv")

In [ ]:
# ============================================================================
# FINAL SUMMARY
# ============================================================================

print("\n" + "="*80)
print("BUNDLE ANALYSIS COMPLETE")
print("="*80)

print("\n📊 KEY FINDINGS:")
print("-" * 80)
print(f"1. Top Bundle Opportunity: {top_bundle['category_1']} + {top_bundle['category_2']}")
print(f"   • Lift: {top_bundle['lift']:.2f}x (strong complementarity)")
print(f"   • Optimal discount: {optimal['discount_pct']:.0f}%")
print(f"   • Expected profit lift: {optimal['profit_lift_pct']:+.1f}%")

print(f"\n2. Customer Behavior:")
print(f"   • Bundle buyers have {clv_lift:.2f}x higher CLV")
print(f"   • Bundle buyers have {orders_lift:.2f}x more orders")
print(f"   • Bundle buyers are {bundled_repeat/unbundled_repeat:.2f}x more likely to return")

print(f"\n3. Revenue Impact:")
print(f"   • Total incremental profit: BRL {total_impact:,.2f}")
print(f"   • Recommended implementation: Phased rollout with A/B testing")

print("\n FILES CREATED:")
print("-" * 80)
print("Data Files:")
print("  • bundle_complementarity.csv")
print("  • bundle_discount_optimization.csv")
print("  • bundle_recommendations.csv")
print("\nVisualizations:")
print("  • bundle_complementarity_heatmap.png")
print("  • bundle_discount_optimization.png")
print("  • customer_segmentation_bundles.png")
print("  • bundle_revenue_impact.png")